# What happens when a vortex pair hits a wall?

Two vortices of opposite sign carry each other toward the upper wall of a square box. We run the same initial flow twice. In one run the fluid can slide along the upper wall with zero tangential stress (free slip). In the other the fluid velocity vanishes there (stationary no slip). How do the vortices move after the encounter, and how much kinetic energy does viscosity remove near the wall?

The figures and movie come with the repository. Run the cells in order with the project’s Python environment to check the initial conditions, play the movie, and recompute the dissipation table from both grid resolutions.

## Two checks on the solver

**Free-slip decay.** A sinusoidal vorticity field has an exact solution in which viscosity reduces the amplitude while preserving the shape. On the right, halving the grid spacing reduces the final field error by a factor of about four. This tests spatial accuracy for a smooth solution.

![Exact mode decay and spatial convergence](figures/walls/free_slip_decay.png)

**Moving no-slip lid.** A lid moving to the right sets the interior fluid in motion. At Reynolds number 100, the two centerline profiles approach the published numerical reference values as the grid is refined. The maximum discrepancy on the finer grid is 0.59% of lid speed. The right-hand panel shows when the computed velocity field settles. [Ghia, Ghia, and Shin, Tables I and II](https://doi.org/10.1016/0021-9991%2882%2990058-4)

![Cavity centerline profiles and approach to steady flow](figures/walls/cavity_profiles.png)

## Change one wall

The collision takes place in a unit square at kinematic viscosity 0.001. All four walls block normal flow. The bottom, left, and right walls use free slip in both runs. The initial streamfunction vanishes near every wall, so the same starting velocity satisfies both upper-wall conditions. We can check the full initial fields using the experiment’s own setup.

In [ ]:
from pathlib import Path

import numpy as np
from IPython.display import Markdown, Video, display
from scipy.integrate import trapezoid

from simulations.run_2d_wall_collision import matched_pair
from simulations.run_2d_wall_dissipation import compare

free, rigid = matched_pair(193)
np.testing.assert_allclose(free.velocity(), rigid.velocity(), atol=1e-12)
np.testing.assert_allclose(free.vorticity(), rigid.vorticity(), atol=1e-12)
for case in (free, rigid):
    print(
        f"{case.walls['top'].kind:9s}  initial kinetic energy = {case.kinetic_energy():.6f}"
    )

## Follow the vortices through impact

The panels share a clock and color scale. Red indicates counterclockwise rotation and blue indicates clockwise rotation. Around simulation time 0.55, look immediately above each core. In the no-slip panel, a thin layer of the opposite sign forms along the wall and curls around the core. By time 0.90 the free-slip cores have traveled farther sideways, while the no-slip cores have moved farther back into the box. The colors saturate at ±75 so the thin layers remain visible. [Open the movie](figures/walls/vortex_wall_collision.mp4)

In [ ]:
movie_path = Path("figures/walls/vortex_wall_collision.mp4")
Video(filename=str(movie_path), embed=True, width=850)

## Where is kinetic energy dissipated?

We integrate the local viscous dissipation density, twice the viscosity times the squared strain-rate tensor norm, over the top sixteenth of the box. This is a fixed physical strip on both grids. Its lower edge falls exactly on a grid line. We sample every 0.01 time units through time 1.20.

![Near-wall dissipation on both grids](figures/walls/vortex_wall_dissipation.png)

Read the three panels from left to right. The first gives the energy lost in the strip **per unit time**. The middle divides that rate by the whole-box dissipation rate at the same instant. The last accumulates the strip loss over time and divides by the **initial kinetic energy**. The no-slip rate peaks at roughly twice the free-slip rate during impact. Later the free-slip rate is higher, while the cumulative loss is still greater in the no-slip case.

### Recompute the four runs

The next cell advances both wall conditions on each grid using the same time stepping as the movie. The last column checks the whole-box energy balance by adding cumulative viscous loss to final kinetic energy and subtracting initial kinetic energy. Neither stationary wall supplies mechanical power, so the continuum residual is zero.

In [ ]:
rows = [
    "| Nodes per side | Upper wall | Peak strip rate | Strip loss / initial energy | Strip loss / total loss | Signed budget residual / initial energy |",
    "|---:|:---|---:|---:|---:|---:|",
]
for nodes in (129, 193):
    times, rates, energies = compare(nodes)
    for index, wall in enumerate(("Free slip", "No slip")):
        total_rate, strip_rate = rates[:, index, :].T
        total_loss = trapezoid(total_rate, x=times)
        strip_loss = trapezoid(strip_rate, x=times)
        initial = energies[0, index]
        residual = (energies[-1, index] + total_loss - initial) / initial
        rows.append(
            f"| {nodes} | {wall} | {strip_rate.max():.5f} | {strip_loss / initial:.2%} "
            f"| {strip_loss / total_loss:.2%} | {residual:+.2%} |"
        )
display(Markdown("\n".join(rows)))

On the fine grid the strip dissipates **17.46% of the initial energy with no slip** and **13.20% with free slip**. The coarse-grid values are 17.30% and 13.18%. The wall effect is larger than the change between these two grids. The fine-grid whole-box budget still has a residual of about 0.51% of initial energy in the no-slip case. These measurements describe this dipole, viscosity, strip width, and time window. [Experiment setup](simulations/run_2d_wall_collision.py) · [Dissipation calculation](simulations/run_2d_wall_dissipation.py)